# Defense 1 — Adversarial Training
## Defending against FGSM/PGD Attacks

**Concept:** Train the network not just on clean images, but also on adversarial
examples generated dynamically during training. This forces the model to learn
robust features that resist gradient-based perturbations.

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set = torch.utils.data.Subset(train_set, range(2000))
test_set = torch.utils.data.Subset(test_set, range(500))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)
print('Data loaded.')

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

def pgd(model, X, y, epsilon=0.2, alpha=0.01, num_steps=10):
    X_adv = X.clone().detach()
    fn = nn.CrossEntropyLoss()
    for _ in range(num_steps):
        X_adv.requires_grad_(True)
        fn(model(X_adv), y).backward()
        with torch.no_grad():
            X_adv = X_adv + alpha * X_adv.grad.sign()
            delta = torch.clamp(X_adv - X, -epsilon, epsilon)
            X_adv = torch.clamp(X + delta, 0, 1).detach()
    return X_adv

## Step 1 — Train Standard Model and Evaluate Vulnerability

In [ ]:
std_model = CNN().to(DEVICE)
opt = optim.Adam(std_model.parameters(), lr=1e-3); fn = nn.CrossEntropyLoss()
print('Training Standard Model...')
for epoch in range(2):
    std_model.train()
    for X, y in train_loader: 
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(std_model(X), y).backward(); opt.step()

std_model.eval()
correct_clean = 0; correct_adv = 0
for X, y in test_loader:
    X, y = X.to(DEVICE), y.to(DEVICE)
    X_adv = pgd(std_model, X, y, epsilon=0.2, num_steps=10)
    with torch.no_grad():
        correct_clean += (std_model(X).argmax(1) == y).sum().item()
        correct_adv += (std_model(X_adv).argmax(1) == y).sum().item()
acc_std_clean = correct_clean / len(test_set)
acc_std_adv   = correct_adv / len(test_set)

std_model.eval()
sample_x, sample_y = test_set[0]
sample_x = sample_x.unsqueeze(0).to(DEVICE)
sample_adv = pgd(std_model, sample_x, torch.tensor([sample_y]).to(DEVICE), epsilon=0.2, num_steps=10)
pred_clean = std_model(sample_x).argmax(1).item()
pred_adv = std_model(sample_adv).argmax(1).item()
fig_before, axes = plt.subplots(1, 2, figsize=(6,3))
axes[0].imshow(sample_x.cpu().squeeze(), cmap='gray')
axes[0].set_title(f'Clean (Pred: {pred_clean}, True: {sample_y})')
axes[0].axis('off')
axes[1].imshow(sample_adv.cpu().squeeze(), cmap='gray')
axes[1].set_title(f'Adv (Pred: {pred_adv}) - Wrong!')
axes[1].axis('off')
plt.suptitle('Before Defense: Standard Model fails on Adv Image')
plt.tight_layout(); plt.savefig('01_before_defense.png', dpi=150); plt.show()

## Step 2 — Adversarial Training

In [ ]:
adv_model = CNN().to(DEVICE)
opt_adv = optim.Adam(adv_model.parameters(), lr=1e-3)
print('Adversarial Training...')
losses_clean = []; losses_adv = []
for epoch in range(2):
    adv_model.train()
    for i, (X, y) in enumerate(train_loader):
        X, y = X.to(DEVICE), y.to(DEVICE)
        X_adv = pgd(adv_model, X, y, epsilon=0.2, num_steps=4) # Faster PGD for training
        
        opt_adv.zero_grad()
        loss_clean = fn(adv_model(X), y)
        loss_adv = fn(adv_model(X_adv), y)
        loss = (loss_clean + loss_adv) / 2
        loss.backward(); opt_adv.step()
        
        if i % 50 == 0:
            losses_clean.append(loss_clean.item()); losses_adv.append(loss_adv.item())

fig_in, ax_in = plt.subplots(figsize=(6,4))
ax_in.plot(losses_clean, label='Clean Loss', color='#27ae60')
ax_in.plot(losses_adv, label='Adv Loss', color='#e74c3c')
ax_in.set_title('In Process: Adversarial Training Loss')
ax_in.legend(); plt.tight_layout(); plt.savefig('01_in_process_defense.png', dpi=150); plt.show()

## Step 3 — Evaluate Robustness

In [ ]:
adv_model.eval()
correct_clean = 0; correct_adv = 0
for X, y in test_loader:
    X, y = X.to(DEVICE), y.to(DEVICE)
    X_adv = pgd(adv_model, X, y, epsilon=0.2, num_steps=10)
    with torch.no_grad():
        correct_clean += (adv_model(X).argmax(1) == y).sum().item()
        correct_adv += (adv_model(X_adv).argmax(1) == y).sum().item()
acc_rob_clean = correct_clean / len(test_set)
acc_rob_adv   = correct_adv / len(test_set)

pred_rob_adv = adv_model(sample_adv).argmax(1).item()
fig_after, ax_after = plt.subplots(figsize=(4,4))
ax_after.imshow(sample_adv.cpu().squeeze(), cmap='gray')
ax_after.set_title(f'Adv (Pred: {pred_rob_adv}, True: {sample_y}) - Correct!')
ax_after.axis('off')
plt.suptitle('After Defense: Robust Model correctly classifies Adv Image')
plt.tight_layout(); plt.savefig('01_after_defense.png', dpi=150); plt.show()